In [1]:
# this code unfolds a variable size array stored in ROOT format into a fixed size pandas dataframe
# by KT 18 July 2024
import uproot
import pandas as pd

def unfold_variable_size_array(file_name, tree_name, size_branch_name, array_features, max_length=None, selection=None):
    # Open the ROOT file
    file = uproot.open(file_name)

    # Access the TTree
    tree = file[tree_name]

    # Extract the size branch
    nSize = tree[size_branch_name].array()

    # Extract all required branches
    arrays = {size_branch_name: nSize}
    for feature in array_features:
        arrays[feature] = tree[feature].array()

    # Apply selection if provided
    if selection:
        mask = eval(selection, {key: arrays[key] for key in arrays})
        for key in arrays:
            arrays[key] = arrays[key][mask]

    # Function to unfold a variable size array into individual columns up to max_length
    def unfold_variable_array(branch_array, nSize, max_length, prefix):
        if max_length is None:
            max_length = max(nSize)
        
        unfolded = {f"{prefix}_{i}": [] for i in range(max_length)}
        
        for event in branch_array:
            for i in range(max_length):
                if i < len(event):
                    unfolded[f"{prefix}_{i}"].append(event[i])
                else:
                    unfolded[f"{prefix}_{i}"].append(float('nan'))
        
        return unfolded

    # Dictionary to hold all unfolded arrays
    unfolded_data = {}
    
    # Extract and unfold each array feature
    for feature in array_features:
        unfolded_feature = unfold_variable_array(arrays[feature], arrays[size_branch_name], max_length, feature)
        unfolded_data.update(unfolded_feature)

    # Add the size branch to the data
    unfolded_data[size_branch_name] = arrays[size_branch_name]

    # Convert to DataFrame
    df = pd.DataFrame(unfolded_data)
    return df

# Example usage
file_name = 'http://theofil.web.cern.ch/theofil/cmsod/files/data.root'
tree_name = 'events'
size_branch_name = 'NJet'


array_features = ['Jet_Px', 'Jet_Py', 'Jet_Pz', 'Jet_E', 'Jet_btag', 'Jet_ID']
max_length = 2
selection = 'NJet>=2'

df = unfold_variable_size_array(file_name, tree_name, size_branch_name, array_features, max_length, selection)
df

,Jet_Px_0,Jet_Px_1,Jet_Py_0,Jet_Py_1,Jet_Pz_0,Jet_Pz_1,Jet_E_0,Jet_E_1,Jet_btag_0,Jet_btag_1,Jet_ID_0,Jet_ID_1,NJet
0,51.975395,-34.074997,1.082405,15.742425,-7.919009,17.878653,53.380703,42.922600,-1.000000,-1.0,True,True,3
1,43.594387,-13.801000,-9.994152,-29.740419,12.846220,-12.826879,48.447979,35.457607,-1.000000,-1.0,True,True,2
2,39.984135,5.111666,44.048626,-33.782871,119.080719,-10.438580,133.310104,36.295273,-1.000000,-1.0,True,True,3
3,45.755638,-15.069008,-71.089729,68.416367,-306.662079,47.942204,318.378601,85.731491,-1.000000,-1.0,True,True,3
4,18.639996,-31.338337,-33.806427,1.699554,66.373062,-31.980953,77.444786,45.087784,-1.000000,-1.0,True,True,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26594,-34.746799,28.346342,23.316643,-17.952951,71.864601,26.360313,83.332863,42.916771,2.766413,-1.0,True,True,2
26595,-44.218636,-28.050138,10.737289,-24.224960,216.854889,-43.493088,221.922562,57.560204,-1.000000,-1.0,True,True,2
26596,-15.815899,-21.016773,-43.644249,-26.615768,-58.733479,-77.256165,75.859154,84.922798,-1.000000,-1.0,True,True,2
26597,-48.930408,33.387009,17.023884,-7.445025,77.781052,7.115584,94.080521,35.330208,3.301559,-1.0,True,True,3


In [2]:
array_features = ['Jet_Px', 'Jet_Py', 'Jet_Pz', 'Jet_E', 'Jet_btag', 'Jet_ID']
max_length = 2
selection = 'NJet>=1'

df = unfold_variable_size_array(file_name, tree_name, size_branch_name, array_features, max_length, selection)
df

,Jet_Px_0,Jet_Px_1,Jet_Py_0,Jet_Py_1,Jet_Pz_0,Jet_Pz_1,Jet_E_0,Jet_E_1,Jet_btag_0,Jet_btag_1,Jet_ID_0,Jet_ID_1,NJet
0,-13.063463,NaN,32.325207,NaN,42.100697,NaN,54.889263,NaN,-1.0,NaN,True,NaN,1
1,51.975395,-34.074997,1.082405,15.742425,-7.919009,17.878653,53.380703,42.922600,-1.0,-1.0,True,True,3
2,43.594387,-13.801000,-9.994152,-29.740419,12.846220,-12.826879,48.447979,35.457607,-1.0,-1.0,True,True,2
3,-36.539494,NaN,-14.458829,NaN,-81.984901,NaN,91.119278,NaN,-1.0,NaN,True,NaN,1
4,64.228813,NaN,68.386971,NaN,371.880096,NaN,383.637115,NaN,-1.0,NaN,True,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93561,15.899570,NaN,35.658783,NaN,36.663506,NaN,54.078747,NaN,-1.0,NaN,True,NaN,1
93562,-9.524969,NaN,-34.514435,NaN,14.504826,NaN,39.167061,NaN,-1.0,NaN,True,NaN,1
93563,37.797947,NaN,-1.518477,NaN,-136.981918,NaN,142.135635,NaN,-1.0,NaN,True,NaN,1
93564,-19.604582,NaN,-37.989887,NaN,-10.661651,NaN,44.590588,NaN,-1.0,NaN,True,NaN,1


In [3]:
array_features = ['Jet_Px', 'Jet_Py', 'Jet_Pz', 'Jet_E', 'Jet_btag', 'Jet_ID']
max_length = 2
selection = 'NJet>=0'

df = unfold_variable_size_array(file_name, tree_name, size_branch_name, array_features, max_length, selection)
df

,Jet_Px_0,Jet_Px_1,Jet_Py_0,Jet_Py_1,Jet_Pz_0,Jet_Pz_1,Jet_E_0,Jet_E_1,Jet_btag_0,Jet_btag_1,Jet_ID_0,Jet_ID_1,NJet
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,-13.063463,NaN,32.325207,NaN,42.100697,NaN,54.889263,NaN,-1.0,NaN,True,NaN,1
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
469379,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
469380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
469381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
469382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
